In [1]:
import pandas as pd
import panel as pn
import matplotlib.pyplot as plt
import folium
from folium.plugins import HeatMap
from dashboard_energia.transformacion.transformacion import transformar_columnas as trans_col, antiguamiento_luminarias as antig_lum    
from folium.plugins import MarkerCluster
from io import BytesIO
import geopandas as gpd
from shapely.geometry import Point
import plotly.graph_objects as go
import plotly.express as px
pn.extension("plotly")


In [2]:
# Cargar datos
df = pd.read_csv("dashboard_energia/data/EQUIPO_AP_LUMINARIA.csv", encoding="utf-8")

C:\Users\ymnl_\AppData\Local\Temp\ipykernel_20164\2065602223.py:2: DtypeWarning: Columns (3,4,5,13) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("dashboard_energia/data/EQUIPO_AP_LUMINARIA.csv", encoding="utf-8")


In [3]:

#tranformamos las columnas
antig_lum(trans_col(df))
print(df.shape[0])



2356768


In [4]:
# Crear geometría de puntos
geometry = [Point(xy) for xy in zip(df["COORDENADA_X"], df["COORDENADA_Y"])]
gdf_luminarias = gpd.GeoDataFrame(df, geometry=geometry, crs="EPSG:4326")  # WGS84

gdf_distritos = gpd.read_file("dashboard_energia/data/DISTRITOS.shp")  # INEI o MIDIS
gdf_distritos = gdf_distritos.to_crs("EPSG:4326")  # Asegura que ambos estén en el mismo CRS

gdf_intersectado = gpd.sjoin(gdf_luminarias, gdf_distritos, how="inner", predicate="intersects")


In [5]:
print(gdf_intersectado.head())
print(gdf_intersectado.columns)

  FECHA_CORTE FECHA_EMISION CODEMP  CODLUMINARIA    CODTRAMOBT CODTRAMOVIA  \
0  2025-06-18    2023-12-31   SEAL  ATO000001581  VBT000108575     1046792   
1  2025-06-18    2023-12-31   SEAL  ATO000001630  VBT000063148     1282305   
2  2025-06-18    2023-12-31   SEAL  ATO000001632  VBT000182767     1282305   
3  2025-06-18    2023-12-31   SEAL  ATO000001659  VBT000113803     1270015   
4  2025-06-18    2023-12-31   SEAL  ATO000001726  VBT000063872     1270175   

                                DESCRIPCION_PASTORAL      PROPIEDAD  \
0  PASTORAL METALICO SIMPLE DE 3,2 m x 1,5 pulg D...  Distribuidora   
1  PASTORAL METALICO SIMPLE DE 3,4 m x 2 pulg DE ...  Distribuidora   
2  PASTORAL METALICO SIMPLE DE 1,5 m x 1,5 pulg D...  Distribuidora   
3  PASTORAL METALICO SIMPLE DE 3,2 m x 1,5 pulg D...  Distribuidora   
4                          CONCRETO SIMPLE DE 1,30 m  Distribuidora   

  FECPUESTASERVICIO SECTOR  ... OBJECTID_1  UBIGEO CODDEP DEPARTAMEN  CODPROV  \
0        2022-11-19    

In [6]:
df_poblacion = pd.read_csv("dashboard_energia/data/dato-poblacion.csv", encoding="ISO-8859-1", sep=";")

df_poblacion["UBIGEO"] = df_poblacion["UBIGEO"].astype(str).str.zfill(6)

df_combinado = gdf_intersectado.merge(df_poblacion, on="UBIGEO", how="left")


In [7]:
print(df_combinado.columns)

print(df_combinado.shape[0])
print(df_combinado.head())

Index(['FECHA_CORTE_x', 'FECHA_EMISION', 'CODEMP', 'CODLUMINARIA',
       'CODTRAMOBT', 'CODTRAMOVIA', 'DESCRIPCION_PASTORAL', 'PROPIEDAD',
       'FECPUESTASERVICIO', 'SECTOR', 'COD_SISTEMA_ELECTRICO',
       'NOMBRE_SISTEMA_ELECTRICO', 'ALIMENTADOR', 'SUBESTACION',
       'COORDENADA_X', 'COORDENADA_Y', 'ANTIGUEDAD', 'geometry', 'index_right',
       'OBJECTID_1', 'UBIGEO', 'CODDEP', 'DEPARTAMEN', 'CODPROV',
       'PROVINCIA_x', 'CODDIST', 'DISTRITO_x', 'CAPITAL', 'FUENTE',
       'FECHA_CORTE_y', 'ANIO', 'DEPARTAMENTO', 'PROVINCIA_y', 'DISTRITO_y',
       'REGION_NATURAL', 'TIPO_MUNICIPALIDAD', 'POB_TOTAL_INEI',
       'POB_URBANA_INEI', 'POB_RURAL_INEI', 'CLASIFICACION_MUNICIPAL_MEF',
       'GENERACION_PER_CAPITA_DOM', 'GENERACION_DOM_URBANA_TDIA',
       'GENERACION_DOM URBANA_TANIO', 'GENERACION_MUN_TANIO',
       'GENERACION_MUN_TDIA', 'GENERACION_PER_CAPITA_MUNICIPAL'],
      dtype='object')
2355562
  FECHA_CORTE_x FECHA_EMISION CODEMP  CODLUMINARIA    CODTRAMOBT CODTRAMOVIA 

In [37]:
#Agrupamos y agregamos
df_grouped = df_combinado.groupby(['UBIGEO', 'DISTRITO_x'], as_index= False).agg({
    'CODLUMINARIA': 'count',
    'ANTIGUEDAD': 'mean',
    'POB_TOTAL_INEI': 'mean',
    'POB_RURAL_INEI': 'mean',
    'TIPO_MUNICIPALIDAD': lambda x: x.mode().iloc[0],
    'CODEMP': lambda x: list(sorted(set(x))),
    'DESCRIPCION_PASTORAL': lambda x: list(sorted(set(x))),
    
}).rename(columns={'CODLUMINARIA': 'N_LUMINARIAS'})

print(df_grouped.head())

   UBIGEO   DISTRITO_x  N_LUMINARIAS  ANTIGUEDAD  POB_TOTAL_INEI  \
0  010101  CHACHAPOYAS          3077    7.250569         40824.0   
1  010102     ASUNCION             8    9.000000           273.0   
2  010103       BALSAS            76    8.315789          1147.0   
3  010104        CHETO            27    1.888889           709.0   
4  010105    CHILIQUIN            21    8.571429           568.0   

   POB_RURAL_INEI TIPO_MUNICIPALIDAD        CODEMP  \
0           965.0         PROVINCIAL        [ELOR]   
1           142.0          DISTRITAL        [ELOR]   
2           869.0          DISTRITAL  [ELNM, ELOR]   
3           264.0          DISTRITAL        [ELOR]   
4           422.0          DISTRITAL        [ELOR]   

                                DESCRIPCION_PASTORAL  
0  [PASTORAL METALICO SIMPLE DE 0,5 m x 1 pulg DE...  
1  [PASTORAL METALICO SIMPLE DE 0,5 m x 1 pulg DE...  
2  [PASTORAL METALICO SIMPLE DE 0,5 m x 1 pulg DE...  
3  [PASTORAL METALICO SIMPLE DE 0,5 m x 1 pulg

In [38]:
bins = [0, 10, 20, 30, 40, float("inf")]
labels = ['0-9 años', '10-19 años', '20-29 años', '30-39 años', '≥40 años']

df_grouped['RANGO_ANTIGUEDAD'] = pd.cut(
    df_grouped['ANTIGUEDAD'],
    bins=bins,
    labels=labels,
    right=False,
    include_lowest=True
)

In [39]:
# Indicadores derivados:
df_grouped['LUMINARIAS_POR_1000HAB'] = df_grouped['N_LUMINARIAS'] / df_grouped['POB_TOTAL_INEI'] * 1000
df_grouped['%POB_RURAL'] = df_grouped['POB_RURAL_INEI'] / df_grouped['POB_TOTAL_INEI'] * 100

print(df_grouped.head())

   UBIGEO   DISTRITO_x  N_LUMINARIAS  ANTIGUEDAD  POB_TOTAL_INEI  \
0  010101  CHACHAPOYAS          3077    7.250569         40824.0   
1  010102     ASUNCION             8    9.000000           273.0   
2  010103       BALSAS            76    8.315789          1147.0   
3  010104        CHETO            27    1.888889           709.0   
4  010105    CHILIQUIN            21    8.571429           568.0   

   POB_RURAL_INEI TIPO_MUNICIPALIDAD        CODEMP  \
0           965.0         PROVINCIAL        [ELOR]   
1           142.0          DISTRITAL        [ELOR]   
2           869.0          DISTRITAL  [ELNM, ELOR]   
3           264.0          DISTRITAL        [ELOR]   
4           422.0          DISTRITAL        [ELOR]   

                                DESCRIPCION_PASTORAL RANGO_ANTIGUEDAD  \
0  [PASTORAL METALICO SIMPLE DE 0,5 m x 1 pulg DE...         0-9 años   
1  [PASTORAL METALICO SIMPLE DE 0,5 m x 1 pulg DE...         0-9 años   
2  [PASTORAL METALICO SIMPLE DE 0,5 m x 1 pulg 

In [40]:
distritos = gpd.read_file('dashboard_energia/data/limites/Limite Distrital INEI 2025 CPV.shp').to_crs('EPSG:32718')
distritos['AREA_KM2'] = distritos['geometry'].area / 1e6

# 2. Reproyectar a lat/lon para visualización
distritos = distritos.to_crs('EPSG:4326')

# 3. Unir con indicadores
df_final = distritos.merge(df_grouped, on='UBIGEO')
df_final['LUMINARIAS_POR_KM2'] = df_final['N_LUMINARIAS'] / df_final['AREA_KM2']

df_final['geometry'] = df_final['geometry'].simplify(0.001, preserve_topology=True)

print(df_final.shape[0])


1837


In [47]:
df_final['DESCRIPCION_PASTORAL'] = df_final['DESCRIPCION_PASTORAL'].apply(
    lambda x: '<br>'.join(x) if isinstance(x, list) else x
)



In [ ]:
import panel as pn
import pandas as pd
import plotly.express as px

pn.extension('plotly')

# Asegurar que CODEMP sea una lista en cada fila
df_final['CODEMP'] = df_final['CODEMP'].apply(lambda x: x if isinstance(x, list) else [x])



# Selectores dinámicos
selector_indicador = pn.widgets.Select(
    name='Indicador',
    options={
        'Antigüedad promedio': 'ANTIGUEDAD',
        'Luminarias por 1000 hab': 'LUMINARIAS_POR_1000HAB',
        'Densidad por km²': 'LUMINARIAS_POR_KM2'
    }
)

selector_distrito = pn.widgets.Select(
    name='Distrito seleccionado',
    options=list(df_final['DISTRITO'].unique()),
    value=list(df_final['DISTRITO'].unique())[0]
)

selector_municipio = pn.widgets.MultiSelect(
    name='Tipo de municipio',
    options=list(df_final['TIPO_MUNICIPALIDAD'].unique()),
    value=list(df_final['TIPO_MUNICIPALIDAD'].unique())
)


# Extraer empresas únicas desde listas
empresas_unicas = sorted(set(
    empresa
    for sublist in df_final['CODEMP']
    for empresa in sublist
))

selector_emp = pn.widgets.MultiSelect(
    name='Empresa',
    options=empresas_unicas,
    value=empresas_unicas
)



# Filtro reactivo
def update_map(indicador, municipios, empresa):
    df_filtrado = df_final[
        df_final['TIPO_MUNICIPALIDAD'].isin(municipios) &
        df_final['CODEMP'].apply(lambda lista: any(e in lista for e in empresa))

    ].copy()

    # Clasificación por rangos de antigüedad
    bins = [0, 10, 20, 30, 40, float("inf")]
    labels = ['0-9 años', '10-19 años', '20-29 años', '30-39 años', '≥40 años']
    df_filtrado['RANGO_ANTIGUEDAD'] = pd.cut(
        df_filtrado['ANTIGUEDAD'],
        bins=bins,
        labels=labels,
        right=False,
        include_lowest=True
    )

    fig = px.choropleth_mapbox(
        df_filtrado,
        geojson=df_filtrado.geometry,
        locations=df_filtrado.index,
        color='RANGO_ANTIGUEDAD',
        hover_name='DISTRITO',
        hover_data={
            'ANTIGUEDAD': True,
            'CODEMP': True,
            'DESCRIPCION_PASTORAL': True,
            'TIPO_MUNICIPALIDAD': True
        },
        center={"lat": -9.19, "lon": -75.015},
        zoom=5,
        color_discrete_map={
            '0-9 años': '#ffffff',
            '10-19 años': "#fcfcb3",
            '20-29 años': '#ff8000',
            '30-39 años': '#ff0000',
            '≥40 años': '#800000'
        },
        mapbox_style="carto-positron"
    )

    fig.update_layout(
        hoverlabel=dict(
            bgcolor='rgba(0,0,0,0.3)',
            font_size=8,
            font_family='Arial',
            font_color='white'
        )
    )

    plot = pn.pane.Plotly(fig, config={'responsive': True})

    def callback(event):
        if event.new and 'points' in event.new:
            distrito = event.new['points'][0]['hovertext']
            selector_distrito.value = distrito

            # Esperar un momento y limpiar el filtro
            def reset_distrito():
                selector_distrito.value = None

            pn.state.add_timeout(1000, reset_distrito)  # 1000 ms = 1 segundo


    plot.param.watch(callback, 'click_data')
    return plot

def update_table(indicador, municipios, empresa, distrito):
    df_filtrado = df_final[
        df_final['TIPO_MUNICIPALIDAD'].isin(municipios) &
        df_final['CODEMP'].apply(lambda lista: any(e in lista for e in empresa)) &
        (df_final['DISTRITO'] == distrito)
    ]

    columnas_mostrar = ['DISTRITO', 'TIPO_MUNICIPALIDAD', 'CODEMP', indicador]
    df_tabla = df_filtrado[columnas_mostrar].copy()

    return pn.widgets.Tabulator(df_tabla, width=600, height=400)

def update_barplot(indicador, municipios, empresa):
    df_filtrado = df_final[
        df_final['TIPO_MUNICIPALIDAD'].isin(municipios) &
        df_final['CODEMP'].apply(lambda lista: any(e in lista for e in empresa))
    ].copy()

    # Explode para convertir listas en filas individuales
    df_exploded = df_filtrado.explode('CODEMP')

    # Agrupar por empresa
    df_bar = df_exploded.groupby('CODEMP')['ANTIGUEDAD'].mean().reset_index()
    df_bar = df_bar.sort_values('ANTIGUEDAD', ascending=False)

    fig = px.bar(
        df_bar,
        x='CODEMP',
        y='ANTIGUEDAD',
        title='Antigüedad promedio por empresa eléctrica',
        labels={'CODEMP': 'Empresa', 'ANTIGUEDAD': 'Antigüedad (años)'},
        color='ANTIGUEDAD',
        color_continuous_scale='Oranges'
    )

    return pn.pane.Plotly(fig, config={'responsive': True})

def update_pastoral_barplot(indicador, municipios, empresa):
    df_filtrado = df_final[
        df_final['TIPO_MUNICIPALIDAD'].isin(municipios) &
        df_final['CODEMP'].apply(lambda lista: any(e in lista for e in empresa))
    ]

    df_pastoral = df_filtrado.groupby('DESCRIPCION_PASTORAL')['ANTIGUEDAD'].mean().reset_index()
    df_pastoral = df_pastoral.sort_values('ANTIGUEDAD', ascending=False)

    fig = px.bar(
        df_pastoral,
        x='DESCRIPCION_PASTORAL',
        y='ANTIGUEDAD',
        title='Antigüedad promedio por descripción pastoral',
        labels={'DESCRIPCION_PASTORAL': 'Descripción Pastoral', 'ANTIGUEDAD': 'Antigüedad (años)'},
        color='ANTIGUEDAD',
        color_continuous_scale='Blues'
    )

    return pn.pane.Plotly(fig, config={'responsive': True})

# Layout del dashboard
dashboard = pn.Column(
    "# 🗺️ Dashboard de cobertura energética",
    pn.Row(selector_indicador, selector_municipio, selector_emp),
    selector_distrito,
    pn.Row(
        pn.panel(pn.bind(update_map, selector_indicador, selector_municipio, selector_emp)),
        pn.panel(pn.bind(update_table, selector_indicador, selector_municipio, selector_emp, selector_distrito))
    ),
    pn.panel(pn.bind(update_barplot, selector_indicador, selector_municipio, selector_emp)),
    pn.panel(pn.bind(update_pastoral_barplot, selector_indicador, selector_municipio, selector_emp))
)

dashboard.servable()
pn.serve(dashboard)


C:\Users\ymnl_\AppData\Local\Temp\ipykernel_20164\1399309656.py:69: DeprecationWarning:

*choropleth_mapbox* is deprecated! Use *choropleth_map* instead. Learn more at: https://plotly.com/python/mapbox-to-maplibre/



Launching server at http://localhost:53346


ERROR:bokeh.server.protocol_handler:error handling message
 message: Message 'PATCH-DOC' content: {'events': [{'kind': 'MessageSent', 'msg_type': 'bokeh_event', 'msg_data': {'type': 'event', 'name': 'plotly_event', 'values': {'type': 'map', 'entries': [['model', {'id': '6d6ef88d-2271-4ef0-9f24-df2021c63580'}], ['data', {'type': 'map', 'entries': [['type', 'click'], ['data', {'type': 'map', 'entries': [['device_state', {'type': 'map', 'entries': [['alt', False], ['ctrl', False], ['meta', False], ['shift', False], ['button', 0], ['buttons', 0]]}], ['selector', None], ['points', [{'type': 'map', 'entries': [['curveNumber', 2], ['pointNumber', 2], ['pointIndex', 2], ['location', 199], ['z', 1], ['hovertext', 'SANTA ROSA'], ['customdata', [20.097472924187727, 'ELNM', 'PASTORAL METALICO SIMPLE DE 0,5 m x 1 pulg DE DIAMETRO', 'DISTRITAL']]]}]]]}]]}]]}}}]} 
 error: AttributeError("'_state' object has no attribute 'add_timeout'")
Traceback (most recent call last):
  File "d:\dataton2025\energia